# 运动会得分谜题：项目数未知，照样能解出来

## 本章要点

这道题的麻烦很具体：题目里没有告诉我们一共办了几项比赛，既然不知道，就从 2 项、3 项、4 项这样一路试上去，第一次解成功的那一个数字，就是真的项目数。

- 不定的“个数”放在外层循环里，一个一个试
- 名次如何换算成得分，`If` 和 `Function` 各写一版
- 用 `m[score_func]` 把 Z3 求出来的函数内容取出来看

> 3 个人开一场运动会，参加者就这 3 个。每个项目里，第 1 名、第 2 名、第 3 名各拿一份分数。分数在所有项目中都一样：第 1 名永远是 X 分，第 2 名永远是 Y 分，第 3 名永远是 Z 分。
>
> 这三个分数是整数，而且满足 `X > Y > Z > 0`。现在全部项目都结束了，结果是这样。
>
> * A 一共拿了 22 分
> * B 在标枪里拿了第 1 名，一共拿了 9 分
> * C 一共拿了 9 分
>
> 现在要知道的是：100 米跑的第 2 名是谁。

※ 这道题参考了 [https://sist8.com/3ath](https://sist8.com/3ath)。

## 难在哪里

- 项目数不明：3 个人的总分摆在眼前，可一共比了几项，题目里并没有交代。
- X、Y、Z 也不明：手上只有 `X > Y > Z > 0` 这一条大小关系。
- 每项的名次同样不明：除了 B 在标枪拿了第 1，其他的名次一概不知。

三个“不明”实际上是同一件事，即值未知的变量。既然是变量，就原样交给 Z3，我们只负责把条件写清楚。其中唯一会改变变量个数的是项目数，因此这一层用循环来试。

## 改变项目数来求解

`try_solve(n)` 干的事很单一：假装比赛有 `n` 项，然后试着往下解。解出来了就打印结果、返回 `True`；解不出来，返回 `False`。

In [1]:
from z3 import *


def try_solve(n_games):
    """假设项目数为 n_games 来求解。有解就显示并返回 True"""
    X, Y, Z = Ints('X Y Z')          # 第 1 名、第 2 名、第 3 名的得分
    solver = Solver()
    solver.add(X > Y, Y > Z, Z > 0)

    # games['A', i] 是“A 在第 i 个项目中的名次”
    games = {}
    for i in range(n_games):
        for p in 'ABC':
            games[p, i] = Int(f'{p}_rank_{i}')
            solver.add(games[p, i] >= 1, games[p, i] <= 3)
        solver.add(Distinct(games['A', i], games['B', i], games['C', i]))

    def score(p):
        """p 的总得分：按名次加上 X、Y、Z"""
        return Sum([If(games[p, i] == 1, X, If(games[p, i] == 2, Y, Z))
                    for i in range(n_games)])

    solver.add(score('A') == 22)
    solver.add(score('B') == 9)
    solver.add(score('C') == 9)
    solver.add(games['B', 0] == 1)   # 第 0 个项目（标枪）中 B 是第 1 名

    if solver.check() != sat:
        print(f"项目数 {n_games}: 无解")
        return False

    m = solver.model()
    print(f"项目数 {n_games}: 第1名={m[X]}分, 第2名={m[Y]}分, 第3名={m[Z]}分")

    ranks = {p: [m[games[p, i]].as_long() for i in range(n_games)] for p in 'ABC'}
    for i in range(n_games):
        name = '标枪' if i == 0 else f'项目{i}'
        print('  ' + name + ': ' + ', '.join(f'{p}=第{ranks[p][i]}名' for p in 'ABC'))
    return True


for n in range(2, 10):
    if try_solve(n):
        break

项目数 2: 无解
项目数 3: 无解
项目数 4: 无解
项目数 5: 第1名=5分, 第2名=2分, 第3名=1分
  标枪: A=第2名, B=第1名, C=第3名
  项目1: A=第1名, B=第3名, C=第2名
  项目2: A=第1名, B=第3名, C=第2名
  项目3: A=第1名, B=第3名, C=第2名
  项目4: A=第1名, B=第3名, C=第2名


2 项、3 项、4 项都解不出结果，到 5 项才第一次出现解，此时三个人的得分是第 1 名 5 分、第 2 名 2 分、第 3 名 1 分(一项 8 分，5 项正好 40 分＝22+9+9)。

再顺着名次看下去：A 在标枪以外的 4 项里全是第 1，B 在标枪以外全是第 3，而 C 在标枪以外的每一项都是第 2，因此 100 米跑的第 2 名就是 C。

单看名次到得分的转换这一步，用 `If` 就能写成“名次是 1 给 X，是 2 给 Y，其余给 Z”。问题在于一嵌套就变得难看，层数一多，读起来先迷路。

## 用 Function 重写

转换也可以做成一张表，交给 `Function()` 保存，原来嵌套的 `If` 全没了，读起来干净。

```python
score_func = Function('score', IntSort(), IntSort())
```

三个参数按顺序看，各自的含义如下。

- `'score'` … 函数的名字，只用来显示
- `IntSort()`(最后一个)… 返回值是什么类型
- `IntSort()`(中间)… 参数是什么类型

这个函数接收一个名次(整数)，返回一个得分(整数)，两头都是整数，因此两个位置都写 `IntSort()`。

表建好了还得填，把“第 1 名 X 分、第 2 名 Y 分、第 3 名 Z 分”这三行对应关系当约束加进去。

```python
solver.add(
    score_func(1) == X,
    score_func(2) == Y,
    score_func(3) == Z,
)
```

加完以后，`score_func(名次)` 直接就是得分。

In [2]:
def try_solve(n_games):
    """用 Function 重写的版本"""
    X, Y, Z = Ints('X Y Z')
    solver = Solver()
    solver.add(X > Y, Y > Z, Z > 0)

    # 把名次 → 得分 的对应关系当作函数持有
    score_func = Function('score', IntSort(), IntSort())
    solver.add(score_func(1) == X, score_func(2) == Y, score_func(3) == Z)

    games = {}
    for i in range(n_games):
        for p in 'ABC':
            games[p, i] = Int(f'{p}_rank_{i}')
            solver.add(games[p, i] >= 1, games[p, i] <= 3)
        solver.add(Distinct(games['A', i], games['B', i], games['C', i]))

    def score(p):
        return Sum([score_func(games[p, i]) for i in range(n_games)])

    solver.add(score('A') == 22)
    solver.add(score('B') == 9)
    solver.add(score('C') == 9)
    solver.add(games['B', 0] == 1)

    if solver.check() != sat:
        print(f"项目数 {n_games}: 无解")
        return False

    m = solver.model()
    print(f"项目数 {n_games}: 第1名={m[X]}分, 第2名={m[Y]}分, 第3名={m[Z]}分")

    ranks = {p: [m[games[p, i]].as_long() for i in range(n_games)] for p in 'ABC'}
    for i in range(n_games):
        name = '标枪' if i == 0 else f'项目{i}'
        print('  ' + name + ': ' + ', '.join(f'{p}=第{ranks[p][i]}名' for p in 'ABC'))

    # 从模型里取出函数本身，就能看到定义好的对应关系
    print('  score_func:', m[score_func])
    return True


for n in range(2, 10):
    if try_solve(n):
        break

项目数 2: 无解
项目数 3: 无解


项目数 4: 无解
项目数 5: 第1名=5分, 第2名=2分, 第3名=1分
  标枪: A=第2名, B=第1名, C=第3名
  项目1: A=第1名, B=第3名, C=第2名
  项目2: A=第1名, B=第3名, C=第2名
  项目3: A=第1名, B=第3名, C=第2名
  项目4: A=第1名, B=第3名, C=第2名
  score_func: [1 -> 5, 3 -> 1, else -> 2]


`m[score_func]` 能把函数内容取出来看。屏幕上那个 `[1 -> 5, 3 -> 1, else -> 2]` 说的是：第 1 名 5 分，第 3 名 1 分，剩下的(也就是第 2 名)2 分。Z3 手里的那张表，就这样完整地显示了出来。

## 小结

| | 结果 |
|---|---|
| 项目数 | 2 项、3 项、4 项都无解，直到 5 项才解得开 |
| 得分 | 第 1 名 5 分、第 2 名 2 分、第 3 名 1 分(满足 X > Y > Z > 0 都满足) |
| 100 米跑的第 2 名 | C(C 在标枪以外的每一项里都是第 2 名) |

- 拿不准的项目个数，放进外层循环，从小到大挨个试一遍
- 名次换得分，`If` 和 `Function` 都能做，表一变大，`Function` 读起来就轻松多了
- 模型定义了哪些内容，用 `m[函数]` 一查就知道，不必再猜